In [4]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path


In [5]:
INPUT_PATH = "../data/processed/train_fe.csv"
OUTPUT_DIR = Path("../data/processed/baselines")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])


In [7]:
df = df.sort_values(["item_id", "date"]).reset_index(drop=True)


In [8]:
HORIZON = 28

train_df = (
    df
    .groupby("item_id")
    .apply(lambda x: x.iloc[:-HORIZON])
    .reset_index(drop=True)
)

valid_df = (
    df
    .groupby("item_id")
    .apply(lambda x: x.iloc[-HORIZON:])
    .reset_index(drop=True)
)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_29436\2039935084.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[:-HORIZON])
C:\Users\ASUS\AppData\Local\Temp\ipykernel_29436\2039935084.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[-HORIZON:])


In [9]:
last_sales = (
    train_df
    .groupby("item_id")["sales"]
    .last()
)

valid_df["pred_naive"] = valid_df["item_id"].map(last_sales)


In [10]:
ma7 = (
    train_df
    .groupby("item_id")["sales"]
    .apply(lambda x: x.tail(7).mean())
)

valid_df["pred_ma7"] = valid_df["item_id"].map(ma7)


In [11]:
ma28 = (
    train_df
    .groupby("item_id")["sales"]
    .apply(lambda x: x.tail(28).mean())
)

valid_df["pred_ma28"] = valid_df["item_id"].map(ma28)


In [12]:
pred_cols = [
    "date", "store_id", "item_id", "sales",
    "pred_naive", "pred_ma7", "pred_ma28"
]

baseline_predictions = valid_df[pred_cols]

baseline_predictions.to_csv(
    OUTPUT_DIR / "baseline_predictions.csv",
    index=False
)


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }



In [16]:
metrics = {
    "Naive": evaluate(valid_df["sales"], valid_df["pred_naive"]),
    "MA_7": evaluate(valid_df["sales"], valid_df["pred_ma7"]),
    "MA_28": evaluate(valid_df["sales"], valid_df["pred_ma28"]),
}


In [18]:
print(metrics)

{'Naive': {'MAE': 1.8058731199925033, 'RMSE': np.float64(4.05117517944311)}, 'MA_7': {'MAE': 1.288925442266116, 'RMSE': np.float64(2.792413527652307)}, 'MA_28': {'MAE': 1.2264568175581154, 'RMSE': np.float64(2.613153428886125)}}


In [17]:
baseline_metrics = (
    pd.DataFrame(metrics)
    .T
    .reset_index()
    .rename(columns={"index": "model"})
)

baseline_metrics.to_csv(
    OUTPUT_DIR / "baseline_metrics.csv",
    index=False
)


## Baseline Model Results

Three baseline forecasting models were evaluated at the item level for a
representative store using a 28-day forecasting horizon.

| Model | MAE | RMSE |
|------|-----|------|
| Naive | 1.81 | 4.05 |
| MA-7 | 1.29 | 2.79 |
| MA-28 | 1.23 | 2.61 |

Moving average models significantly outperform the naive baseline by smoothing
intermittent demand patterns. The 28-day moving average achieves the lowest error,
indicating that incorporating longer historical context improves forecast accuracy.

These results establish a strong baseline and motivate the use of machine learning
models to further capture non-linear patterns and external effects.
